# Librerias

In [1]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
from io import StringIO
import requests
import json 
import time 
from tqdm import tqdm  
import kagglehub
import requests
import csv
import os
import shutil
from itertools import combinations
import sqlite3

pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

C:\Users\JuanSebastiánArbelae\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Datos Olist

In [2]:
def cargar_olist():
    archivos = [
        "olist_customers_dataset.csv", "olist_geolocation_dataset.csv",
        "olist_order_items_dataset.csv", "olist_order_payments_dataset.csv",
        "olist_order_reviews_dataset.csv", "olist_orders_dataset.csv",
        "olist_products_dataset.csv", "olist_sellers_dataset.csv",
        "product_category_name_translation.csv"
    ]
    
    ruta_base = kagglehub.dataset_download("olistbr/brazilian-ecommerce", force_download=True)
    
    datasets_cargados = {}

    for archivo in archivos:
        ruta_full = os.path.join(ruta_base, archivo)
        nombre_df = f"{archivo.replace('olist_', '').replace('_dataset', '').replace('.csv', '')}"
        
        try:
            df = pd.read_csv(ruta_full, encoding='latin1')
            datasets_cargados[nombre_df] = df
            print(f"✅{nombre_df}")
        except Exception as e:
            print(f"❌{archivo}: {e}")
            
    return datasets_cargados

In [3]:
datos = cargar_olist()

100%|█████████████████████████████████████████████████████████████████████████████| 42.6M/42.6M [00:08<00:00, 5.28MB/s]

Extracting files...


✅customers
✅geolocation
✅order_items
✅order_payments
✅order_reviews
✅orders
✅products
✅sellers
✅product_category_name_translation


# Web Scraping

Caso especial

In [4]:
datos_libros = []
paginas_a_raspar = 50

for pagina in range(1, paginas_a_raspar + 1):
    
    url_paginada = f"https://books.toscrape.com/catalogue/page-{pagina}.html"
    
    try:
        respuesta = requests.get(url_paginada, timeout=10)
        
        if respuesta.status_code == 200:
            sopa = BeautifulSoup(respuesta.text, 'html.parser')
            
            libros = sopa.find_all('article', class_='product_pod')
            
            for libro in libros:
                titulo = libro.find('h3').find('a')['title']
                
                # El precio
                precio = libro.find('p', class_='price_color').text
                
                # El stock (si dice "In stock" o no)
                stock = libro.find('p', class_='instock availability').text.strip()
                
                datos_libros.append({
                    'Pagina_Origen': pagina, 
                    'Titulo': titulo,
                    'Precio_Crudo': precio,
                    'Disponibilidad': stock
                })
                
            print(f"✅ Página {pagina} completada. ({len(libros)} libros extraídos)")
        else:
            print(f"⚠️ Error en la página {pagina}. Código: {respuesta.status_code}")
            
        time.sleep(1)

    except Exception as e:
        print(f"❌ Error al conectar con la página {pagina}: {e}")

# CONSOLIDACIÓN
df_libros = pd.DataFrame(datos_libros)
print(f"\n ¡Scraping completado! Tenemos {len(df_libros)} libros en nuestra base de datos.")

✅ Página 1 completada. (20 libros extraídos)
✅ Página 2 completada. (20 libros extraídos)
✅ Página 3 completada. (20 libros extraídos)
✅ Página 4 completada. (20 libros extraídos)
✅ Página 5 completada. (20 libros extraídos)
✅ Página 6 completada. (20 libros extraídos)
✅ Página 7 completada. (20 libros extraídos)
✅ Página 8 completada. (20 libros extraídos)
✅ Página 9 completada. (20 libros extraídos)
✅ Página 10 completada. (20 libros extraídos)
✅ Página 11 completada. (20 libros extraídos)
✅ Página 12 completada. (20 libros extraídos)
✅ Página 13 completada. (20 libros extraídos)
✅ Página 14 completada. (20 libros extraídos)
✅ Página 15 completada. (20 libros extraídos)
✅ Página 16 completada. (20 libros extraídos)
✅ Página 17 completada. (20 libros extraídos)
✅ Página 18 completada. (20 libros extraídos)
✅ Página 19 completada. (20 libros extraídos)
✅ Página 20 completada. (20 libros extraídos)
✅ Página 21 completada. (20 libros extraídos)
✅ Página 22 completada. (20 libros extraído

In [5]:
df_libros.head()

,Pagina_Origen,Titulo,Precio_Crudo,Disponibilidad
0,1,A Light in the Attic,Â£51.77,In stock
1,1,Tipping the Velvet,Â£53.74,In stock
2,1,Soumission,Â£50.10,In stock
3,1,Sharp Objects,Â£47.82,In stock
4,1,Sapiens: A Brief History of Humankind,Â£54.23,In stock


Caso especial ajustado

In [6]:
datos_libros = []
paginas_a_raspar = 50

tiempo_inicio = time.time()

for pagina in tqdm(range(1, paginas_a_raspar + 1), desc="Descargando catálogo", unit="pág"):
    
    url_paginada = f"https://books.toscrape.com/catalogue/page-{pagina}.html"
    
    try:
        respuesta = requests.get(url_paginada, timeout=10)
        
        if respuesta.status_code == 200:
            sopa = BeautifulSoup(respuesta.text, 'html.parser')
            
            libros = sopa.find_all('article', class_='product_pod')
            
            for libro in libros:
                titulo = libro.find('h3').find('a')['title']
                precio = libro.find('p', class_='price_color').text
                stock = libro.find('p', class_='instock availability').text.strip()
                
                datos_libros.append({
                    'Pagina_Origen': pagina, 
                    'Titulo': titulo,
                    'Precio_Crudo': precio,
                    'Disponibilidad': stock
                })
            
        else:
            print(f"\n⚠️ Error en la página {pagina}. Código: {respuesta.status_code}")
            
        time.sleep(1)

    except Exception as e:
        print(f"\n❌ Error al conectar con la página {pagina}: {e}")

tiempo_fin = time.time()
duracion_segundos = round(tiempo_fin - tiempo_inicio, 2)
duracion_minutos = round(duracion_segundos / 60, 2)

df_libros = pd.DataFrame(datos_libros)

print(f"\n¡Scraping completado con éxito!")
print(f" Tiempo total de ejecución: {duracion_segundos} segundos ({duracion_minutos} minutos).")
print(f" Tenemos {len(df_libros)} libros en nuestra base de datos.")

Descargando catálogo: 100%|███████████████████████████████████████████████████████████| 50/50 [01:26<00:00,  1.73s/pág]


¡Scraping completado con éxito!
 Tiempo total de ejecución: 86.37 segundos (1.44 minutos).
 Tenemos 1000 libros en nuestra base de datos.


Usuario y contraseña

In [7]:
session = requests.Session()

login_url = "http://quotes.toscrape.com/login"
target_url = "http://quotes.toscrape.com/" 

print("Obteniendo la página de login...")
response = session.get(login_url)
soup = BeautifulSoup(response.text, 'html.parser')

# Buscamos el campo oculto (input) llamado 'csrf_token'
csrf_token = soup.find('input', {'name': 'csrf_token'})['value']
print(f"Token de seguridad interceptado: {csrf_token}")

payload = {
    'csrf_token': csrf_token,
    'username': 'estudiante_etl', 
    'password': 'password123'     
}

print("Enviando credenciales...")
login_response = session.post(login_url, data=payload)

protected_page = session.get(target_url)

if "Logout" in protected_page.text:
    print("✅ ¡Login exitoso! Acceso concedido a los datos protegidos.")
    
    soup_data = BeautifulSoup(protected_page.text, 'html.parser')
    
    quotes = soup_data.find_all('span', class_='text')
    for quote in quotes[:3]: 
        print(f"- {quote.text}")
        
else:
    print("❌ Error de autenticación. Revisa el payload o las credenciales.")

Obteniendo la página de login...
Token de seguridad interceptado: XpmCNGBolfKAnkSUTuvbQieqcVOyagxWdrYJEDsHFtwjIhLRzZPM
Enviando credenciales...
✅ ¡Login exitoso! Acceso concedido a los datos protegidos.
- “The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”
- “It is our choices, Harry, that show what we truly are, far more than our abilities.”
- “There are only two ways to live your life. One is as though nothing is a miracle. The other is as though everything is a miracle.”


# API

In [8]:
print("Iniciando extracción desde la API del Banco Central (Frankfurter)...")

url_api = "https://api.frankfurter.app/latest?from=GBP&to=BRL,USD"

try:
    respuesta = requests.get(url_api, timeout=10)
    
    if respuesta.status_code == 200:
        datos_json = respuesta.json()
        
        fecha_actualizacion = datos_json['date']
        tasa_brl = datos_json['rates']['BRL']
        tasa_usd = datos_json['rates']['USD']
        
        print("¡Conexión exitosa!")
        print(f" Fecha de cotización: {fecha_actualizacion}")
        print(f" 1 Libra (£) = {tasa_brl} Reales (R$)")
        print(f" 1 Libra (£) = {tasa_usd} Dólares ($)")
        
        df_tasas = pd.DataFrame([{
            'Moneda_Origen': 'GBP',
            'Tasa_BRL': tasa_brl,
            'Tasa_USD': tasa_usd,
            'Fecha_ETL': fecha_actualizacion
        }])
        
    else:
        print(f"⚠️ Error en el servidor de la API. Código HTTP: {respuesta.status_code}")

except requests.exceptions.RequestException as e:
    print(f"❌ Error crítico de conexión: {e}")

Iniciando extracción desde la API del Banco Central (Frankfurter)...
¡Conexión exitosa!
 Fecha de cotización: 2026-04-30
 1 Libra (£) = 6.7292 Reales (R$)
 1 Libra (£) = 1.3509 Dólares ($)


In [9]:
df_tasas

,Moneda_Origen,Tasa_BRL,Tasa_USD,Fecha_ETL
0,GBP,6.7292,1.3509,2026-04-30


# Transformacion

In [10]:
tasa_gbp_usd = df_tasas['Tasa_USD'].iloc[0] 
tasa_gbp_brl = df_tasas['Tasa_BRL'].iloc[0] 

tasa_brl_usd = tasa_gbp_usd / tasa_gbp_brl 

In [11]:
df_libros['Precio_Original'] = df_libros['Precio_Crudo'].str.extract(r'(\d+\.\d+)').astype(float)

df_libros['Precio_USD'] = (df_libros['Precio_Original'] * tasa_gbp_usd).round(2)

df_libros['Origen_Datos'] = 'Web_Scraping_Competencia'
df_libros['Categoria'] = 'libros_competencia'

nuevos_ids = ['SCRP-' + str(i).zfill(4) for i in range(1, len(df_libros) + 1)]

df_libros.insert(0, 'id_producto', nuevos_ids)

In [12]:
df_libros.head()

,id_producto,Pagina_Origen,Titulo,Precio_Crudo,Disponibilidad,Precio_Original,Precio_USD,Origen_Datos,Categoria
0,SCRP-0001,1,A Light in the Attic,Â£51.77,In stock,51.77,69.94,Web_Scraping_Competencia,libros_competencia
1,SCRP-0002,1,Tipping the Velvet,Â£53.74,In stock,53.74,72.60,Web_Scraping_Competencia,libros_competencia
2,SCRP-0003,1,Soumission,Â£50.10,In stock,50.10,67.68,Web_Scraping_Competencia,libros_competencia
3,SCRP-0004,1,Sharp Objects,Â£47.82,In stock,47.82,64.60,Web_Scraping_Competencia,libros_competencia
4,SCRP-0005,1,Sapiens: A Brief History of Humankind,Â£54.23,In stock,54.23,73.26,Web_Scraping_Competencia,libros_competencia


In [13]:
df_productos = datos['products']

todas_las_categorias = df_productos['product_category_name'].dropna().unique()

categorias_libros = [categoria for categoria in todas_las_categorias if 'livro' in categoria.lower()]

print(f"\n✅ Se encontraron {len(categorias_libros)} categorías relacionadas con libros:")
for categoria in categorias_libros:
    print(f"   📚 {categoria}")

print("\n📊 Conteo de productos por categoría:")
conteo = df_productos[df_productos['product_category_name'].isin(categorias_libros)]['product_category_name'].value_counts()
print(conteo)


✅ Se encontraron 3 categorías relacionadas con libros:
   📚 livros_interesse_geral
   📚 livros_tecnicos
   📚 livros_importados

📊 Conteo de productos por categoría:
product_category_name
livros_interesse_geral    216
livros_tecnicos           123
livros_importados          31
Name: count, dtype: int64


In [14]:
df_productos_olist = datos['products']
df_ventas_olist = datos['order_items']

categorias_libros = ['livros_interesse_geral', 'livros_tecnicos', 'livros_importados']
df_productos_filtrado = df_productos_olist[df_productos_olist['product_category_name'].isin(categorias_libros)]

df_olist_libros = pd.merge(
    df_productos_filtrado, 
    df_ventas_olist, 
    on='product_id', 
    how='inner'
)

df_olist_libros['Precio_USD'] = (df_olist_libros['price'] * tasa_brl_usd).round(2)

df_olist_libros['Origen_Datos'] = 'Olist_Interno'

In [15]:
df_olist_libros.head()

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm,order_id,order_item_id,seller_id,shipping_limit_date,price,freight_value,Precio_USD,Origen_Datos
0,8c9faf6aabee9b2a30b83f92d786f6de,livros_interesse_geral,50.0,2913.0,3.0,1000.0,24.0,5.0,17.0,f713f69364d66b2020e04ec8892d1285,1,fc5a0d7a310a7a41abf86a458585ff2b,2017-06-08 16:45:25,117.90,19.07,23.67,Olist_Interno
1,c04b8195870068ff5c495f269a0bc993,livros_tecnicos,54.0,1162.0,1.0,1100.0,29.0,3.0,21.0,3860898db238028285b1e05b5ab31238,1,ba143b05f0110f0dc71ad71b4466ce92,2018-07-24 08:55:29,48.99,19.53,9.83,Olist_Interno
2,ad19bca6e0f1919433d7e21d534ddb50,livros_interesse_geral,42.0,1109.0,1.0,900.0,18.0,25.0,16.0,5b74abebfd5bc1d137d7ade71c5cfd36,1,527801b552d0077ffd170872eb49683b,2018-09-03 20:35:18,44.90,13.89,9.01,Olist_Interno
3,ad19bca6e0f1919433d7e21d534ddb50,livros_interesse_geral,42.0,1109.0,1.0,900.0,18.0,25.0,16.0,616dfda2aa81fd23f5e8f47619f650be,1,527801b552d0077ffd170872eb49683b,2018-08-28 13:30:51,44.90,13.89,9.01,Olist_Interno
4,ad19bca6e0f1919433d7e21d534ddb50,livros_interesse_geral,42.0,1109.0,1.0,900.0,18.0,25.0,16.0,b7276795d572042799508b017fb3cf16,1,527801b552d0077ffd170872eb49683b,2018-04-25 02:15:15,39.90,18.23,8.01,Olist_Interno


In [16]:
olist_limpio = df_olist_libros[['product_id', 'product_category_name', 'Precio_USD', 'Origen_Datos']].copy()
olist_limpio.columns = ['ID_Producto', 'Categoria', 'Precio_USD', 'Origen_Datos']

scraping_limpio = df_libros[['id_producto', 'Categoria', 'Precio_USD', 'Origen_Datos']].copy()
scraping_limpio.columns = ['ID_Producto', 'Categoria', 'Precio_USD', 'Origen_Datos'] 

df_catalogo_maestro = pd.concat([olist_limpio, scraping_limpio], ignore_index=True)

In [17]:
scraping_limpio.head()

,ID_Producto,Categoria,Precio_USD,Origen_Datos
0,SCRP-0001,libros_competencia,69.94,Web_Scraping_Competencia
1,SCRP-0002,libros_competencia,72.60,Web_Scraping_Competencia
2,SCRP-0003,libros_competencia,67.68,Web_Scraping_Competencia
3,SCRP-0004,libros_competencia,64.60,Web_Scraping_Competencia
4,SCRP-0005,libros_competencia,73.26,Web_Scraping_Competencia


In [18]:
df_catalogo_maestro.sample(5)

,ID_Producto,Categoria,Precio_USD,Origen_Datos
438,e2cac69b319c0f8a21dbf04b925121bf,livros_interesse_geral,8.01,Olist_Interno
803,f35927953ed82e19d06ad3aac2f06353,livros_interesse_geral,23.09,Olist_Interno
584,7bdcad1796b53a8f633c1cfd82102efe,livros_tecnicos,17.86,Olist_Interno
1719,SCRP-0840,libros_competencia,57.02,Web_Scraping_Competencia
400,0d368e3a4bd2ebf60c31c214da108e3b,livros_interesse_geral,6.01,Olist_Interno


# Carga

In [19]:
def enmascarar_id(seller_id):
    id_str = str(seller_id)
    return "****-****-****-" + id_str[-4:] if len(id_str) > 4 else "****"

In [20]:
ventas_a_cargar = df_olist_libros[['order_id', 'order_item_id', 'product_id', 'seller_id', 'price']].copy()
ventas_a_cargar['seller_id'] = ventas_a_cargar['seller_id'].apply(enmascarar_id)

In [43]:
conn = sqlite3.connect('Test3.db')
cursor = conn.cursor()

cursor.execute("PRAGMA foreign_keys = ON;")

cursor.execute('''
CREATE TABLE IF NOT EXISTS olist_productos (
    product_id TEXT PRIMARY KEY,
    product_category_name TEXT
);
''')

cursor.execute('''
CREATE TABLE IF NOT EXISTS olist_ventas (
    order_id TEXT,
    order_item_id INTEGER,
    product_id TEXT,
    seller_id TEXT, -- Este es el campo que acabamos de enmascarar
    price REAL,
    PRIMARY KEY (order_id, order_item_id),
    FOREIGN KEY (product_id) REFERENCES olist_productos(product_id)
);
''')

cursor.execute('''
CREATE TABLE IF NOT EXISTS competencia_libros (
    id_producto TEXT PRIMARY KEY,   
    categoria TEXT,
    precio_usd REAL,
    origen_datos TEXT
);
''')

cursor.execute('''
CREATE TABLE IF NOT EXISTS catalogo_maestro (
    id_producto TEXT PRIMARY KEY,
    categoria TEXT,
    precio_usd REAL,
    origen_datos TEXT
);
''')
conn.commit()

conn.close()

In [29]:
conn = sqlite3.connect('Test.db')
cursor = conn.cursor()
try:
    productos_a_cargar = df_productos_filtrado[['product_id', 'product_category_name']]
    productos_a_cargar.to_sql('olist_productos', conn, if_exists='append', index=False)
    print("✅ Tabla 'olist_productos' cargada.")

    ventas_a_cargar.to_sql('olist_ventas', conn, if_exists='append', index=False)
    print("✅ Tabla 'olist_ventas' cargada (Optimizada y enmascarada).")

    scraping_limpio.to_sql('competencia_libros', conn, if_exists='append', index=False)
    print("✅ Tabla 'competencia_libros' cargada.")

    df_catalogo_maestro.to_sql('catalogo_maestro', conn, if_exists='append', index=False)
    print("✅ Tabla 'catalogo_maestro' cargada exitosamente.")

except sqlite3.IntegrityError as e:
    print(f"\n❌ Error de Integridad (Revisar PK/FK): {e}")
except Exception as e:
    print(f"\n❌ Error general de carga: {e}")

finally:
    conn.close()

✅ Tabla 'olist_productos' cargada.
✅ Tabla 'olist_ventas' cargada (Optimizada y enmascarada).
✅ Tabla 'competencia_libros' cargada.
✅ Tabla 'catalogo_maestro' cargada exitosamente.


In [27]:
cantidad_antes = len(df_catalogo_maestro)

df_catalogo_maestro = df_catalogo_maestro.sort_values(by='Precio_USD', ascending=False)

df_catalogo_maestro = df_catalogo_maestro.drop_duplicates(subset=['ID_Producto'], keep='first')

df_catalogo_maestro = df_catalogo_maestro.reset_index(drop=True)

cantidad_despues = len(df_catalogo_maestro)

print(f"🧹 Se comprimieron {cantidad_antes - cantidad_despues} filas duplicadas.")
print(f"📈 Regla aplicada exitosamente: Se conservó únicamente el PRECIO MÁXIMO de cada producto.")

🧹 Se comprimieron 510 filas duplicadas.
📈 Regla aplicada exitosamente: Se conservó únicamente el PRECIO MÁXIMO de cada producto.


In [30]:
print("Iniciando Carga Incremental Inteligente...")

conn = sqlite3.connect('Test.db')
cursor = conn.cursor()

try:
    df_catalogo_maestro.to_sql('staging_catalogo', conn, if_exists='replace', index=False)
    print("   ⏳ Datos subidos a la tabla temporal de staging...")

    cursor.execute('''
        INSERT OR IGNORE INTO catalogo_maestro (id_producto, categoria, precio_usd, origen_datos)
        SELECT ID_Producto, Categoria, Precio_USD, Origen_Datos 
        FROM staging_catalogo;
    ''')
    
    filas_nuevas = cursor.rowcount
    conn.commit()
    print(f"   ✅ Traslado exitoso: Se insertaron {filas_nuevas} libros NUEVOS al catálogo.")

    cursor.execute("DROP TABLE staging_catalogo;")
    print("   🧹 Tabla temporal destruida.")

except Exception as e:
    print(f"\n❌ Error en la carga incremental: {e}")
finally:
    conn.close()

Iniciando Carga Incremental Inteligente...
   ⏳ Datos subidos a la tabla temporal de staging...
   ✅ Traslado exitoso: Se insertaron 0 libros NUEVOS al catálogo.
   🧹 Tabla temporal destruida.


In [31]:
# --- SIMULACIÓN DE NUEVOS DATOS ---

nuevo_libro_scraping = pd.DataFrame([{
    'ID_Producto': 'SCRP-9999', 
    'Categoria': 'libros_competencia', 
    'Precio_USD': 99.99, 
    'Origen_Datos': 'Web_Scraping_Competencia'
}])

nuevo_producto_olist = pd.DataFrame([{
    'ID_Producto': 'NUEVO-PRODUCTO-123', 
    'Categoria': 'livros_tecnicos', 
    'Precio_USD': 150.00, 
    'Origen_Datos': 'Olist_Interno'
}])

df_simulado_maestro = pd.concat([df_catalogo_maestro, nuevo_libro_scraping, nuevo_producto_olist], ignore_index=True)

print(f"📊 DataFrame preparado: {len(df_simulado_maestro)} registros (incluye 2 nuevos y miles repetidos).")

📊 DataFrame preparado: 1372 registros (incluye 2 nuevos y miles repetidos).


In [32]:
conn = sqlite3.connect('Test.db')
cursor = conn.cursor()

try:
    df_simulado_maestro.to_sql('staging_catalogo', conn, if_exists='replace', index=False)
    
    cursor.execute('''
        INSERT OR IGNORE INTO catalogo_maestro (id_producto, categoria, precio_usd, origen_datos)
        SELECT ID_Producto, Categoria, Precio_USD, Origen_Datos 
        FROM staging_catalogo;
    ''')
    
    filas_nuevas = cursor.rowcount
    conn.commit()
    
    print(f"RESULTADO DE LA CARGA:")
    print(f"   - Registros procesados en total: {len(df_simulado_maestro)}")
    print(f"   - Registros NUEVOS insertados: {filas_nuevas}")
    print(f"   - Registros REPETIDOS ignorados: {len(df_simulado_maestro) - filas_nuevas}")

    cursor.execute("DROP TABLE staging_catalogo;")

except Exception as e:
    print(f"❌ Error: {e}")
finally:
    conn.close()

🚀 RESULTADO DE LA CARGA:
   - Registros procesados en total: 1372
   - Registros NUEVOS insertados: 2
   - Registros REPETIDOS ignorados: 1370


In [33]:
conn = sqlite3.connect('Test.db')

query = "SELECT * FROM catalogo_maestro WHERE id_producto IN ('SCRP-9999', 'NUEVO-PRODUCTO-123')"
df_verificacion = pd.read_sql(query, conn)

conn.close()

if len(df_verificacion) == 2:
    print("✅ Los registros nuevos se cargaron sin duplicar el resto del catálogo.")
    display(df_verificacion)
else:
    print("⚠️ Algo falló, no se encuentran los registros nuevos.")

✅ Los registros nuevos se cargaron sin duplicar el resto del catálogo.


,id_producto,categoria,precio_usd,origen_datos
0,NUEVO-PRODUCTO-123,livros_tecnicos,150.00,Olist_Interno
1,SCRP-9999,libros_competencia,99.99,Web_Scraping_Competencia


Actualización de registos

In [35]:
id_a_modificar = df_catalogo_maestro.iloc[0]['ID_Producto']
precio_anterior = df_catalogo_maestro.iloc[0]['Precio_USD']

print(f" Modificando el producto {id_a_modificar}")
print(f" Precio original: ${precio_anterior}")

nuevo_precio = round(precio_anterior * 1.5, 2)
df_simulado_maestro.loc[df_simulado_maestro['ID_Producto'] == id_a_modificar, 'Precio_USD'] = nuevo_precio

print(f"Nuevo precio simulado: ${nuevo_precio}")

 Modificando el producto b0c7a71c8620bd389e240f63a507dc50...
 Precio original: $180.66
   Nuevo precio simulado: $270.99


In [ ]:
conn = sqlite3.connect('Test.db')
    
query = f"""SELECT * FROM catalogo_maestro WHERE id_producto = '{id_a_modificar}';"""
    
df_resultado = pd.read_sql(query, conn)
print(df_resultado.head())
conn.close()

In [36]:
print("🔄 Iniciando Carga Incremental")

conn = sqlite3.connect('Test.db')
cursor = conn.cursor()

try:
    df_simulado_maestro.to_sql('staging_catalogo', conn, if_exists='replace', index=False)
    
    cursor.execute('''
        REPLACE INTO catalogo_maestro (id_producto, categoria, precio_usd, origen_datos)
        SELECT ID_Producto, Categoria, Precio_USD, Origen_Datos 
        FROM staging_catalogo;
    ''')
    
    conn.commit()
    print("✅ Proceso de actualización (UPSERT) completado exitosamente.")
    print("   - Los registros nuevos se insertaron.")
    print("   - Los registros existentes actualizaron sus precios/categorías.")

    cursor.execute("DROP TABLE staging_catalogo;")

except Exception as e:
    print(f"❌ Error en la carga: {e}")
finally:
    conn.close()

🔄 Iniciando Carga Incremental
✅ Proceso de actualización (UPSERT) completado exitosamente.
   - Los registros nuevos se insertaron.
   - Los registros existentes actualizaron sus precios/categorías.


In [41]:
conn = sqlite3.connect('Test.db')

id_nuevo = 'b0c7a71c8620bd389e240f63a507dc50'
    
query = f"""SELECT * FROM catalogo_maestro WHERE id_producto = '{id_a_modificar}';"""

df_resultado = pd.read_sql(query, conn)
conn.close()

df_resultado.head()

,id_producto,categoria,precio_usd,origen_datos
0,b0c7a71c8620bd389e240f63a507dc50,livros_interesse_geral,270.99,Olist_Interno
